In [ ]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
from scipy import stats
import plotnine as p9
from tqdm.auto import tqdm
import copy
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

from src.clustering.settings import clinical_endpoints, final_cluster_name, rename_cols_dict

plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
results = pd.read_csv(os.path.join("results_v2_permodality", "clustering", "cox_cluster_clinical.csv"))
results["cols"] = results["cols"].apply(lambda x:str([rename_cols_dict[var] if var in rename_cols_dict.keys() else var for var in eval(x)]))
results["cols_eval"] = results["cols"].apply(eval)
results["aucs_eval"] = results["aucs"].str.replace("nan", "np.nan").apply(eval)
results["times_eval"] = results["times"].apply(eval)
results["endpoint"] = results["endpoint"].apply(lambda x: clinical_endpoints[x]["description"])
endpoints = [endpoint["description"] for endpoint in clinical_endpoints.values()]

In [ ]:
summary = (
    results.groupby(["endpoint", "i_col", "ncols"], as_index=False)
      .agg(
          cols=("cols", "first"),
          metric_mean=("c_uno", "mean"),
          metric_std=("c_uno", "std"),
          adjp=("adjp", "mean"),
      )
)
summary["metric_mean_std"] = summary.apply(lambda row: f"{row['metric_mean']:.2f}±{row['metric_std']:.2f}", axis=1)
encoded_summary = summary['cols'].apply(eval).explode().str.get_dummies().groupby(level=0).max()
encoded_summary = summary.join(encoded_summary)
cols = eval(encoded_summary.loc[summary["ncols"].idxmax(), "cols"])
encoded_summary = encoded_summary[["endpoint", "cols", "ncols", "metric_mean", "metric_std", "adjp", "metric_mean_std"] + cols]
encoded_summary = pd.melt(
    encoded_summary, 
    id_vars=['cols', "endpoint", "metric_mean", "metric_std", "ncols", "metric_mean_std", "adjp"], 
    value_vars=cols, 
    var_name='Column', 
    value_name='Status'
)
encoded_summary["Status"] = encoded_summary["Status"].apply(lambda x:"Yes" if x else "No")
encoded_summary["model_type"] = encoded_summary["ncols"].apply(lambda x: "Univariate models" if x == 1 else "Multivariate models")

In [ ]:
p_hm = []
topn = 6
model_types = ["Univariate models", "Multivariate models"]
for idx, endpoint in enumerate(endpoints):
    for model_type in model_types:
        df_to_plot = encoded_summary[encoded_summary["model_type"].eq(model_type) & encoded_summary["endpoint"].eq(endpoint)].sort_values("metric_mean", ascending=False)
        cats = df_to_plot.iloc[:len(cols)*topn][df_to_plot["Status"].eq("Yes")].groupby("Column")["metric_mean"]
        if model_type == model_types[0]:
            tag = "a"
            cats = cats.mean()
        else:
            tag = "b"
            cats = cats.count()
        cats = cats.sort_values().index.to_list()
        cats = [col for col in cols if col not in cats] + cats
        cats = cats[::-1]
        df_to_plot["Column"] = pd.Categorical(df_to_plot["Column"], categories=cats, ordered=True)
        cats = df_to_plot.groupby("cols")["metric_mean"].mean().sort_values().index.to_list()
        cats = cats[::-1]
        df_to_plot["cols"] = "Top" + (pd.Categorical(df_to_plot["cols"], categories=cats, ordered=True).codes + 1).astype(str)
        df_to_plot["cols"] = pd.Categorical(df_to_plot["cols"], categories=df_to_plot["cols"].unique()[::-1], ordered=True)
        row_values = df_to_plot.drop_duplicates("cols")[["cols", "metric_mean", "metric_mean_std", "metric_std", "adjp"]].sort_values("metric_mean", ascending=False).iloc[:topn].iloc[::-1]
        df_to_plot = df_to_plot.sort_values(["metric_mean", "cols", "Column"], ascending=[False, False, False]).iloc[:len(cols)*topn]
        row_values["Significant"] = np.where(row_values["adjp"].le(0.05), "adjp<=0.05", "adjp>0.05")

        # top_mod = df_to_plot[df_to_plot["cols"].isin(["Top1", "Top2"]) & df_to_plot["Status"].eq("Yes")]
        # top1_mod = top_mod[df_to_plot["cols"].eq("Top1")]
        # top1_mods = top1_mod["Column"].unique()
        # top1_mod = results[
        #     results["endpoint"].eq(endpoint) &
        #     results.apply(lambda x: all([mod in x["cols_eval"] for mod in top1_mods]) and (x["ncols"] == len(top1_mods)), axis=1)
        # ]
        # top2_mod = top_mod[df_to_plot["cols"].eq("Top2")]
        # top2_mods = top2_mod["Column"].unique()
        # top2_mod = results[
        #     results["endpoint"].eq(endpoint) &
        #     results.apply(lambda x: all([mod in x["cols_eval"] for mod in top2_mods]) and (x["ncols"] == len(top2_mods)), axis=1)
        # ]
        # print(endpoint, model_type, stats.kruskal(top1_mod["c_uno"], top2_mod["c_uno"])[1], stats.levene(top1_mod["c_uno"], top2_mod["c_uno"])[1])
        
        p = (
            p9.ggplot(df_to_plot, p9.aes("Column", "cols", fill="Status")) 
            + p9.geom_tile(p9.aes(width=0.95, height=0.95))
            + p9.geom_text(
                data=row_values,
                mapping=p9.aes(
                    x=len(cols) + 0.6,
                    y="cols",
                    label="metric_mean_std",
                    color="Significant",
                ),
                inherit_aes=False, ha="left"
            )
            + p9.facet_wrap("endpoint")
            + p9.theme_classic()
            + p9.labs(x="", fill = "Included")
            + p9.theme(axis_title_y=p9.element_blank(), axis_line_y=p9.element_blank(), strip_background=p9.element_blank(),
                       axis_ticks_y=p9.element_blank(), axis_text_x=p9.element_text(angle=30),
                       axis_line_x=p9.element_blank(), axis_ticks_x=p9.element_blank())
            + p9.scale_fill_manual(breaks=["No", "Yes"], values=['#f0f0f0', '#4a4a4a'])
            + p9.scale_color_manual(['red', 'black'])
            + p9.annotate("text", x=7.42, y=topn+.42, label="Uno's C-index")
            + p9.scale_x_discrete(labels=lambda x: [label.replace(" ", "\n") for label in x], expand=(0, 0, 0, 1.1))
            + p9.scale_y_discrete(expand=(0, 0, 0, 0.15))
        )
        if idx == 0:
            legend_position = "top"
            p = p + p9.labs(tag=tag, title=model_type, color="") 
            if model_type == model_types[0]:
                p = p + p9.guides(fill="none")
            else:
                p = p + p9.guides(color="none")
        else:
            legend_position = "none"
        p = p + p9.theme(legend_position=legend_position, legend_box_spacing=0.01)
        if model_type == model_types[1]:
            p = p + p9.theme(axis_text_y=p9.element_blank())
        p_hm.append(p)

(p_hm[0] | p_hm[1]) / (p_hm[2] | p_hm[3]) / (p_hm[4] | p_hm[5]) / (p_hm[6] | p_hm[7])

In [ ]:
xvars = eval(results.loc[results["ncols"].idxmax()]["cols"])
results_xvars = []
for endpoint in tqdm(endpoints):
    for xvar in xvars:
        xvar_ress = results[(results["endpoint"] == endpoint) & (results["ncols"]>1) & (results["cols_eval"].apply(lambda x: xvar in x))]
        for cols in xvar_ress["cols"].unique():
            xvar_res_with = results[(results["endpoint"] == endpoint) & (results["cols"].eq(cols))]
            cols = str([c for c in eval(cols) if c != xvar])
            xvar_res_without = results[(results["endpoint"] == endpoint) & (results["cols"].eq(cols))]
            for fold in xvar_res_with["fold"].unique():
                diff_perf = xvar_res_with[xvar_res_with["fold"] == fold]["c_uno"].iloc[0] - xvar_res_without[xvar_res_without["fold"] == fold]["c_uno"].iloc[0]
                results_xvars_dict = {
                    "endpoint": endpoint,
                    "var": xvar,
                    "vars_with": xvar_res_with["cols"].iloc[0],
                    "vars_without": xvar_res_without["cols"].iloc[0],
                    "nvars_with": xvar_res_with["ncols"].iloc[0],
                    "nvars_without": xvar_res_without["ncols"].iloc[0],
                    "diff": diff_perf,
                }
                results_xvars.append(results_xvars_dict)
            
results_xvars = pd.DataFrame(results_xvars)
grouped_results_xvars = results_xvars.groupby(["endpoint", "var", "vars_without", "nvars_without"], as_index=False)["diff"].mean() \
    .sort_values(["endpoint", "diff"], ascending=False)

In [ ]:
stats_grouped_results_xvars = copy.deepcopy(grouped_results_xvars)
pval_comp_df = []
for endpoint in endpoints:
    stats_grouped_results_xvars_endp = stats_grouped_results_xvars[stats_grouped_results_xvars["endpoint"].eq(endpoint)]
    varref = stats_grouped_results_xvars_endp.groupby("var")["diff"].mean().sort_values(ascending=False).index
    other_var = varref[1]
    varref = varref[0]
    varref_perf = stats_grouped_results_xvars_endp[stats_grouped_results_xvars_endp["var"].eq(varref)]["diff"]
    other_var_perf = stats_grouped_results_xvars_endp[stats_grouped_results_xvars_endp["var"].eq(other_var)]["diff"]
    p = stats.kruskal(varref_perf, other_var_perf)[1]
    pval_dict = {
        "endpoint": endpoint,
        "varref": varref,
        "other_var": other_var,
        "p": p,
    }
    pval_comp_df.append(pval_dict)
pval_comp_df = pd.DataFrame(pval_comp_df)
pval_comp_df["p"] = multipletests(pval_comp_df["p"], method='fdr_bh')[1]
pval_comp_df = pval_comp_df[pval_comp_df["p"].lt(0.05)]
pval_comp_df["y_start"] = -0.044
pval_comp_df["y_end"] = pval_comp_df["y_start"] + 0.015
pval_comp_df["y_label"] = pval_comp_df["y_start"] - 0.02
pval_comp_df["sig"] = pval_comp_df["p"].apply(lambda x: f"***" if x <= 0.001 else ("**" if x <= 0.01 else ("*" if x > 0.01 else "x")))
pval_comp_df["endpoint"] = pd.Categorical(pval_comp_df["endpoint"], categories=endpoints, ordered=True)

In [ ]:
x,y = "var", "diff"
facet_cols = "endpoint"
grouped_results_xvars[facet_cols] = pd.Categorical(grouped_results_xvars[facet_cols], categories=endpoints, ordered=True)
custom_order = grouped_results_xvars.groupby([facet_cols, x], as_index=False)[y].mean().sort_values([facet_cols, y], ascending=[True, True])[x].unique()
grouped_results_xvars[x] = pd.Categorical(grouped_results_xvars[x], categories=custom_order, ordered=True)
pval_comp_df["x_label"] = pval_comp_df.apply(
    lambda x: (np.argmax(custom_order == x["varref"]) + np.argmax(custom_order == x["other_var"]))/2 + 1,
    axis=1
)
p_vars = (
    p9.ggplot(grouped_results_xvars.sort_values(by=y), p9.aes(x=x, y=y, fill=x, color=x, shape=x))
    + p9.geom_jitter(show_legend=False, size=2.5)
    + p9.geom_segment(
            p9.aes(
                 x=p9.stage(x, after_stat="x-.6"),
                 xend=p9.stage(x, after_stat="x+.6"),
                 yend=p9.after_stat("y")
            ),
            stat="summary",
            fun_y=np.mean,
            size=1.5,
            show_legend=False,
        )
    + p9.facet_wrap(facet_cols, nrow=1)
    + p9.geom_hline(yintercept=0., color="red", linetype="dashed", size=0.75)
    + p9.geom_segment(p9.aes(x='varref', xend='other_var', y='y_start', yend='y_start'), data=pval_comp_df, color='black', inherit_aes=False)
    + p9.geom_segment(p9.aes(x='varref', xend='varref', y='y_start', yend='y_end'), data=pval_comp_df, color='black', inherit_aes=False)
    + p9.geom_segment(p9.aes(x='other_var', xend='other_var', y='y_start', yend='y_end'), data=pval_comp_df, color='black', inherit_aes=False)
    + p9.geom_text(p9.aes(x='x_label', y='y_label', label='sig'), data=pval_comp_df, inherit_aes=False)
    + p9.coord_flip()
    + p9.theme_classic()
    + p9.theme(strip_background=p9.element_blank(), axis_title_y=p9.element_blank())
    + p9.scale_color_brewer(type="qualitative", palette="Set2", direction=-1)
    + p9.scale_fill_brewer(type="qualitative", palette="Set2", direction=-1)
    + p9.labs(y="Δ Uno's C-Index", tag="d", title="Feature importance (permutation score)")
    + p9.scale_x_discrete(labels=lambda x: [label.replace(" ", "\n") for label in x])
)
p_vars

In [ ]:
grouped_results_mods = results[
    results["ncols"].eq(results["ncols"].max()) |
    results["cols"].eq(f"['{final_cluster_name}']") |
    (results["ncols"].eq(results["ncols"].max()-1) & results["cols_eval"].apply(lambda x: final_cluster_name not in x))
]
stats_grouped_results_mods = copy.deepcopy(grouped_results_mods)
names = grouped_results_mods.groupby(["endpoint", "cols"], as_index=False)[["c_uno", "ncols"]].mean()
clinical_vars = names[names["ncols"].eq(names["ncols"].max()-1)]["cols"].iloc[0]
all_vars = names[names["ncols"].eq(names["ncols"].max())]["cols"].iloc[0]
rename_vars_dict = {
    f"['{final_cluster_name}']" : final_cluster_name,
    clinical_vars : "Clinical",
    all_vars : f"Clinical + {final_cluster_name}",
}
grouped_results_mods["cols"] = grouped_results_mods["cols"].apply(lambda x: rename_vars_dict[x])
grouped_results_mods = (
    grouped_results_mods.groupby(["endpoint", "cols"], as_index=False)
      .agg(
          metric_mean=("c_uno", "mean"),
          metric_std=("c_uno", "std"),
      )
)


stats_grouped_results_mods["cols"] = stats_grouped_results_mods["cols"].apply(lambda x: rename_vars_dict[x])
max_cols = results["ncols"].max()
pval_comp_df = []
for endpoint in endpoints:
    stats_grouped_results_mods_endp = stats_grouped_results_mods[stats_grouped_results_mods["endpoint"].eq(endpoint)]
    mm_name = stats_grouped_results_mods_endp[stats_grouped_results_mods_endp["ncols"].eq(max_cols)]["cols"].unique()
    assert len(mm_name) == 1
    mm_name = mm_name[0]
    for mod in stats_grouped_results_mods_endp[stats_grouped_results_mods_endp["ncols"].lt(max_cols)]["cols"].unique():
        mm_perf = stats_grouped_results_mods_endp[stats_grouped_results_mods_endp["cols"].eq(mm_name)]["c_uno"]
        um_perf = stats_grouped_results_mods_endp[stats_grouped_results_mods_endp["cols"].eq(mod)]["c_uno"]
        p = stats.kruskal(mm_perf, um_perf)[1]
        pval_dict = {
            "endpoint": endpoint,
            "mm_name": mm_name,
            "um_name": mod,
            "p": p,
        }
        pval_comp_df.append(pval_dict)
pval_comp_df = pd.DataFrame(pval_comp_df)
pval_comp_df["p"] = multipletests(pval_comp_df["p"], method='fdr_bh')[1].round(3)
# pval_comp_df = pval_comp_df[pval_comp_df["p"].lt(0.05)]
pval_comp_df["y_start"] = pval_comp_df["um_name"].apply(lambda x: 0.86 if x == "Clinical" else 0.81)
pval_comp_df["y_end"] = pval_comp_df["y_start"] - 0.015
pval_comp_df["y_label"] = pval_comp_df["y_start"] + 0.01
pval_comp_df["sig"] = pval_comp_df["p"].apply(lambda x: f"***" if x <= 0.001 else ("**" if x <= 0.01 else ("*" if x <= 0.05 else "")))
pval_comp_df["endpoint"] = pd.Categorical(pval_comp_df["endpoint"], categories=endpoints, ordered=True)
pval_comp_df["x_label"] = pval_comp_df["um_name"].apply(lambda x: 1.5 if x == "Clinical" else 2)

In [ ]:
x,y = "cols", "metric_mean"
facet_cols = "endpoint"
grouped_results_mods[facet_cols] = pd.Categorical(grouped_results_mods[facet_cols], categories=endpoints, ordered=True)
custom_order = grouped_results_mods.groupby([facet_cols, x], as_index=False)[y].mean().sort_values([facet_cols, y], ascending=[True, False])[x].unique()
grouped_results_mods[x] = pd.Categorical(grouped_results_mods[x], categories=[f"Clinical + {final_cluster_name}", "Clinical", final_cluster_name], ordered=True)
increased_grouped_results_mods = grouped_results_mods.copy()
increased_grouped_results_mods["pos_y"] = increased_grouped_results_mods.apply(lambda x: 
    increased_grouped_results_mods[increased_grouped_results_mods["endpoint"].eq(x["endpoint"])]["metric_mean"].min(), axis=1)
increased_grouped_results_mods["increased"] = increased_grouped_results_mods.apply(lambda x:
        x["metric_mean"] / increased_grouped_results_mods[increased_grouped_results_mods["endpoint"].eq(x["endpoint"])]["metric_mean"].min(), axis=1)
increased_grouped_results_mods["increased"] = increased_grouped_results_mods["increased"].sub(1).mul(100).round().astype(int).astype(str).add("%")
for endpoint in endpoints:
    idx_drop = increased_grouped_results_mods[increased_grouped_results_mods["endpoint"].eq(endpoint)]["metric_mean"].idxmin()
    increased_grouped_results_mods = increased_grouped_results_mods.drop(idx_drop)
increased_grouped_results_mods["pos_x"] = increased_grouped_results_mods["cols"].cat.codes.add(1).add(0.075)
increased_grouped_results_mods["pos_y_text"] = increased_grouped_results_mods["metric_mean"] - 0.015
increased_grouped_results_mods = increased_grouped_results_mods[increased_grouped_results_mods["increased"].str[:-1].astype(int).gt(0)]
p_mods = (
    p9.ggplot(grouped_results_mods.sort_values(by=x), p9.aes(x=x, y=y, fill=x))
    + p9.geom_col(show_legend=False)
    + p9.geom_errorbar(
        p9.aes(ymin=f'{y}-metric_std', ymax=f'{y}+metric_std'), 
        width=0.2,               # Width of the error bar caps
        color='black'
    )
    + p9.facet_wrap(facet_cols, nrow=1)
    + p9.geom_hline(yintercept=0.5, color="red", linetype="dashed", size=0.75)
    + p9.geom_text(
        data=increased_grouped_results_mods,
        mapping=p9.aes(
            x=x,
            y="pos_y_text",
            label="increased"
        ),
        inherit_aes=False, nudge_x=0.395
    )
    + p9.geom_segment(
        data=increased_grouped_results_mods,
        mapping=p9.aes(
            x="pos_x",
            xend="pos_x",
            y="pos_y",
            yend="metric_mean",
        ), arrow=p9.arrow(length=0.03, type="closed")
    )
    + p9.geom_segment(p9.aes(x='mm_name', xend='um_name', y='y_start', yend='y_start'), data=pval_comp_df, color='black', inherit_aes=False)
    + p9.geom_segment(p9.aes(x='mm_name', xend='mm_name', y='y_start', yend='y_end'), data=pval_comp_df, color='black', inherit_aes=False)
    + p9.geom_segment(p9.aes(x='um_name', xend='um_name', y='y_start', yend='y_end'), data=pval_comp_df, color='black', inherit_aes=False)
    + p9.geom_text(p9.aes(x='x_label', y='y_label', label='sig'), data=pval_comp_df, inherit_aes=False)
    + p9.theme_classic()
    + p9.theme(strip_background=p9.element_blank())
    + p9.scale_fill_brewer(type="qualitative", palette="Pastel1", direction=1)
    + p9.labs(x="", y= "Uno's C-Index", tag="e", title="Unimodal versus multimodal performance")
    + p9.scale_x_discrete(labels=lambda x: [label.replace(" ", "\n") for label in x])
)
p_mods

In [ ]:
results_overtime = []
for endpoint in endpoints:
    best_comb = summary[summary["endpoint"] == endpoint].sort_values(by="metric_mean", ascending=False)["cols"].iloc[0]
    best_comb = eval(best_comb)
    best_comb = results[
        results["fold"].lt(200) &
        (
            results["ncols"].eq(1) & results["cols_eval"].apply(lambda x: not pd.Index(best_comb).intersection(pd.Index(x)).empty) |
            results["ncols"].eq(len(best_comb)) & results["cols_eval"].apply(lambda x: len(pd.Index(best_comb).intersection(pd.Index(x))) == len(best_comb))
        )
    ]
    for cols in best_comb["cols"].unique():
        line = best_comb[best_comb["cols"].eq(cols)]
        cols = eval(cols)
        for i, row in line.iterrows():
            for t,auc in zip(row["times_eval"], row["aucs_eval"]):
                rovertime_dict = {
                    "endpoint": endpoint,
                    "cols": " + ".join(cols) if len(cols) > 1 else cols[0],
                    "Time (months)": t,
                    "AUC": auc,
                }
                results_overtime.append(rovertime_dict)
results_overtime = pd.DataFrame(results_overtime)
results_overtime = results_overtime[results_overtime["endpoint"].isin(endpoints[:2])]
results_overtime["endpoint"] = pd.Categorical(results_overtime["endpoint"], categories=endpoints[:2], ordered=True)

In [ ]:
x,y = "Time (months)", "AUC"
facet_cols = "endpoint"
custom_order = results_overtime.groupby([facet_cols, "cols"], as_index=False)[y].mean().sort_values([facet_cols, y], ascending=[True, False])["cols"].unique().tolist()
results_overtime["cols"] = pd.Categorical(results_overtime["cols"], categories=custom_order, ordered=True)
p_aucs = (
    p9.ggplot(results_overtime, p9.aes(x=x, y=y, fill="cols", color="cols"))
    + p9.geom_smooth(method="loess")
    + p9.facet_wrap(facet_cols)
    + p9.geom_hline(yintercept=0.5, color="red", linetype="dashed", size=0.75)
    + p9.theme_classic()
    + p9.theme(legend_position="top", legend_title=p9.element_blank(), strip_background=p9.element_blank(),
               legend_background=p9.element_blank(), legend_box_spacing=0.01)
    + p9.scale_fill_brewer(type="qualitative", palette="Set1", direction=1)
    + p9.scale_color_brewer(type="qualitative", palette="Set1", direction=1)
    + p9.guides(fill=p9.guide_legend(nrow=2), color=p9.guide_legend(nrow=2))
    + p9.labs(tag="c", title="Time-dependent discrimination")
)
p_aucs

In [ ]:
composition = ((p_hm[0] | p_hm[1]) / (p_hm[2] | p_hm[3]) / (p_hm[4] | p_hm[5]) / (p_hm[6] | p_hm[7])) | (p_aucs / p_vars / p_mods) + p9.theme(figure_size=(19, 14))
composition

In [ ]:
composition.save(os.path.join("figures", "Figure5.svg"))
composition.save(os.path.join("figures", "Figure5.pdf"))